In [33]:
"""
transform_data.py
=================
ETL Step 2: Read raw_gold_data.csv, compute derived features,
split into 4 DataFrames matching the MySQL schema.
Output: table_daily_prices.csv, table_macro_indicators.csv,
        table_etf_flows.csv, table_energy_mining.csv
"""

import pandas as pd
import numpy as np

INPUT_FILE = "raw_gold_data.csv"

MACRO_COLS = [
    "Fed_Rate", "Treasury_10Y", "Treasury_2Y",
    "CPI_Index", "Unemployment", "M2_Supply", "PPI",
]

# column name → yfinance symbol (used in build_daily_prices for the symbol column)
COL_TO_SYMBOL: dict[str, str] = {
    "Gold_Price":      "GC=F",
    "Silver_Price":    "SI=F",
    "Platinum_Price":  "PL=F",
    "Palladium_Price": "PA=F",
    "DXY_Index":       "DX-Y.NYB",
    "SP500":           "^GSPC",
    "VIX_Index":       "^VIX",
    "USD_EUR":         "EURUSD=X",
    "USD_CNY":         "USDCNY=X",
    "USD_RUB":         "RUB=X",
}


def load_raw() -> pd.DataFrame:
    df = pd.read_csv(INPUT_FILE, index_col="Date", parse_dates=True)
    print(f"Loaded: {len(df)} rows, {df.shape[1]} columns")
    return df

In [34]:
def fill_macro_gaps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Forward-fill FRED macro columns.
    FRED publishes monthly/quarterly; ffill propagates the last known value
    across subsequent trading days until the next release.
    """
    existing = [c for c in MACRO_COLS if c in df.columns]
    df[existing] = df[existing].ffill()
    return df

In [35]:
def compute_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute all derived columns in-place. Returns the same DataFrame."""

    # ── macro_indicators ──────────────────────────────────────────────────────

    # YoY inflation: CPI % change over ~252 trading days
    # CPI is monthly so we ffill first to get a daily series before pct_change
    if "CPI_Index" in df.columns:
        df["Inflation_YoY"] = df["CPI_Index"].ffill().pct_change(252) * 100

    # Fisher real rate: nominal 10Y yield minus realized inflation
    if {"Treasury_10Y", "Inflation_YoY"}.issubset(df.columns):
        df["Real_Rate_Calc"] = df["Treasury_10Y"] - df["Inflation_YoY"]

    # Yield curve spread: negative = inversion = recession signal
    if {"Treasury_10Y", "Treasury_2Y"}.issubset(df.columns):
        df["Yield_Curve_10Y2Y"] = df["Treasury_10Y"] - df["Treasury_2Y"]

    # ── etf_flows ─────────────────────────────────────────────────────────────

    # GLD premium to spot: positive = ETF trades above physical gold
    if {"GLD_Price", "Gold_Price"}.issubset(df.columns):
        rolling_ratio = (df["GLD_Price"] / df["Gold_Price"]).rolling(252, min_periods=20).median()
        df["GLD_Gold_Premium"] = (df["GLD_Price"] / (df["Gold_Price"] * rolling_ratio) - 1) * 100

    # ETF inflow signal: price above 20-day MA → institutional accumulation
    if "GLD_Price" in df.columns:
        df["ETF_Signal"] = (df["GLD_Price"] > df["GLD_Price"].rolling(20).mean()).astype(int)

    # ── energy_mining ─────────────────────────────────────────────────────────

    # AISC proxy: 50% copper + 30% WTI + 20% PPI/100
    # Weights approximate share of all-in sustaining cost components
    if {"Copper_Price", "WTI_Oil", "Gold_Price"}.issubset(df.columns):
        base = df[df.index.year == 2000]
        wti_base    = base["WTI_Oil"].mean()
        copper_base = base["Copper_Price"].mean()

        wti_clipped = df["WTI_Oil"].clip(lower=1.0)
        df["Mining_Cost_Index"] = (
            0.60 * (wti_clipped / wti_base) +
            0.40 * (df["Copper_Price"] / copper_base)
        )

        # Margin: how much faster gold price grew vs mining costs since 2000
        gold_base = df["Gold_Price"][df.index.year == 2000].mean()
        df["Gold_Mining_Margin"] = (df["Gold_Price"] / gold_base) / df["Mining_Cost_Index"]

        # Bull signal: gold outpacing costs by >2x relative to 2000 baseline
        df["Mining_Bull_Signal"] = (df["Gold_Mining_Margin"] > 2.0).astype(int)

    return df

In [36]:
def build_daily_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build long-format table for daily_prices.
    One row = one asset × one date.
    ETF and energy columns are excluded (handled by separate builders).
    """
    price_cols = [c for c in COL_TO_SYMBOL if c in df.columns]

    frames = []
    for col in price_cols:
        tmp = df[[col]].dropna().reset_index()
        tmp.columns = ["price_date", "close_price"]
        tmp["symbol"] = COL_TO_SYMBOL[col]
        frames.append(tmp)

    result = (
        pd.concat(frames, ignore_index=True)
        [["price_date", "symbol", "close_price"]]
        .sort_values(["price_date", "symbol"])
        .reset_index(drop=True)
    )
    print(f"  daily_prices:     {len(result):>7} rows")
    return result


daily_prices = build_daily_prices(compute_derived_features(fill_macro_gaps(load_raw())))
daily_prices.tail()

Loaded: 6982 rows, 22 columns
  daily_prices:       62499 rows


,price_date,symbol,close_price
62494,2026-04-16,RUB=X,75.492355
62495,2026-04-16,SI=F,78.606003
62496,2026-04-16,USDCNY=X,6.818000
62497,2026-04-16,^GSPC,7041.279785
62498,2026-04-16,^VIX,17.940001


In [37]:
def build_macro_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build wide-format table for macro_indicators.
    One row per date. Drop dates where Fed_Rate is missing
    (pre-2000 FRED backfill artifacts).
    """
    cols = [
        "Fed_Rate", "Treasury_10Y", "Treasury_2Y",
        "CPI_Index", "Unemployment", "M2_Supply",
        "Inflation_YoY", "Real_Rate_Calc", "Yield_Curve_10Y2Y",
    ]
    cols = [c for c in cols if c in df.columns]

    result = (
        df[cols]
        .dropna(subset=["Fed_Rate"])
        .rename_axis("indicator_date")
        .reset_index()
        .sort_values("indicator_date")
        .reset_index(drop=True)
    )
    print(f"  macro_indicators: {len(result):>7} rows")
    return result

In [38]:
def build_etf_flows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build long-format table for etf_flows.
    One row = one ETF (GLD or IAU) × one date.
    GLD_Gold_Premium and ETF_Signal only computed for GLD;
    IAU rows get 0 / 0 as neutral placeholders.
    """
    ETF_MAP = {"GLD_Price": "GLD", "IAU_Price": "IAU"}

    frames = []
    for col, symbol in ETF_MAP.items():
        if col not in df.columns:
            continue

        tmp = df[[col]].dropna().reset_index()
        tmp.columns = ["flow_date", "etf_price"]
        tmp["symbol"] = symbol

        if symbol == "GLD" and "GLD_Gold_Premium" in df.columns:
            extras = df[["GLD_Gold_Premium", "ETF_Signal"]].reset_index()
            extras.columns = ["flow_date", "gld_gold_premium", "etf_signal"]
            tmp = tmp.merge(extras, on="flow_date", how="left")
        else:
            tmp["gld_gold_premium"] = 0.0
            tmp["etf_signal"] = 0

        frames.append(tmp)

    result = (
        pd.concat(frames, ignore_index=True)
        [["flow_date", "symbol", "etf_price", "gld_gold_premium", "etf_signal"]]
        .assign(
            etf_signal=lambda x: x["etf_signal"].fillna(0).astype(int),
            gld_gold_premium=lambda x: x["gld_gold_premium"].fillna(0.0),
        )
        .sort_values(["flow_date", "symbol"])
        .reset_index(drop=True)
    )
    print(f"  etf_flows:        {len(result):>7} rows")
    return result

In [39]:
def build_energy_mining(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build wide-format table for energy_mining.
    One row per date. Requires WTI_Oil, Mining_Cost_Index,
    and Gold_Mining_Margin to all be non-null.
    """
    cols = [
        "WTI_Oil", "Brent_Oil", "Copper_Price",
        "Mining_Cost_Index", "Gold_Mining_Margin", "Mining_Bull_Signal",
    ]
    cols = [c for c in cols if c in df.columns]

    result = (
        df[cols]
        .dropna(subset=["WTI_Oil", "Mining_Cost_Index", "Gold_Mining_Margin"])
        .rename_axis("record_date")
        .reset_index()
        .sort_values("record_date")
        .reset_index(drop=True)
    )
    print(f"  energy_mining:    {len(result):>7} rows")
    return result

In [40]:
def transform_data() -> None:
    """Run full Step 2: load → fill gaps → derive features → split → save."""
    print(f"=== transform_data | input: {INPUT_FILE} ===\n")

    df = load_raw()
    df = fill_macro_gaps(df)
    df = compute_derived_features(df)

    print("\nBuilding tables:")
    dp  = build_daily_prices(df)
    mi  = build_macro_indicators(df)
    etf = build_etf_flows(df)
    em  = build_energy_mining(df)

    outputs = {
        "table_daily_prices.csv":     dp,
        "table_macro_indicators.csv": mi,
        "table_etf_flows.csv":        etf,
        "table_energy_mining.csv":    em,
    }
    print("\nSaving:")
    for fname, frame in outputs.items():
        frame.to_csv(fname, index=False)
        print(f"  {fname}")

    print("\nDone. Next step: load_to_mysql.py")


transform_data()

=== transform_data | input: raw_gold_data.csv ===

Loaded: 6982 rows, 22 columns

Building tables:
  daily_prices:       62499 rows
  macro_indicators:    6982 rows
  etf_flows:          10722 rows
  energy_mining:       6429 rows

Saving:
  table_daily_prices.csv
  table_macro_indicators.csv
  table_etf_flows.csv
  table_energy_mining.csv

Done. Next step: load_to_mysql.py
